# ETL IV: Load

Follow along with the lesson page on the course site. Each heading below matches a section of the lesson, and its code is already filled in: read it, then run it.

## Setup

In [ ]:
import polars as pl

## Step 3: Connect From Python

### Keeping credentials out of your code

In [ ]:
import os

import polars as pl
from dotenv import find_dotenv, load_dotenv
from sqlalchemy import URL, create_engine, text

load_dotenv(find_dotenv(usecwd=True))

url = URL.create(
    "postgresql+psycopg2",
    username=os.environ["PGUSER"],
    password=os.environ["PGPASSWORD"],
    host=os.environ["PGHOST"],
    port=int(os.environ["PGPORT"]),
    database="course_etl",
)

engine = create_engine(url)

### Testing the connection

In [ ]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database(), current_user"))
    print(result.fetchone())

## Step 4: Load the Tables

### Load order

In [ ]:
LOAD_ORDER = [
    "terms",
    "subjects",
    "buildings",
    "courses",              # references subjects
    "sections",             # references terms, courses
    "meetings",             # references sections, buildings
    "instructors",
    "section_instructors",  # references sections, instructors
]

tables = {
    name: pl.read_csv(f"data/tables/{name}.csv", infer_schema=False)
    for name in LOAD_ORDER
}

### One transaction for the whole load

In [ ]:
with engine.begin() as conn:
    conn.execute(text("TRUNCATE " + ", ".join(LOAD_ORDER)))

    for name in LOAD_ORDER:
        tables[name].write_database(name, connection=conn, if_table_exists="append")
        print(f"{name:<22}{len(tables[name]):>6,} rows")

## Step 5: Verify the Load

### Row counts

In [ ]:
with engine.connect() as conn:
    for name in LOAD_ORDER:
        in_db = conn.execute(text(f"SELECT COUNT(*) FROM {name}")).scalar()
        print(f"{name:<22}{in_db:>6,} in database  {len(tables[name]):>6,} in DataFrame")

### Query it the way an analyst would

In [ ]:
query = """
    SELECT s.subject_code, s.course_number, s.section, s.course_title,
           m.weekdays, m.start_time, m.end_time,
           b.building_description AS building, m.room,
           i.instructor_first_name || ' ' || i.instructor_last_name AS instructor
    FROM sections AS s
    INNER JOIN meetings AS m
        ON m.term_code = s.term_code AND m.crn = s.crn
    LEFT JOIN buildings AS b
        ON b.building_code = m.building_code
    INNER JOIN section_instructors AS si
        ON si.term_code = s.term_code AND si.crn = s.crn
    INNER JOIN instructors AS i
        ON i.instructor_id = si.instructor_id
    WHERE s.subject_code = 'DAT' AND s.course_number = '153'
"""

pl.read_database(query, connection=engine)

In [ ]:
query = """
    SELECT subject_code, course_number, section, course_title,
           enrollment_current, enrollment_max,
           enrollment_max - enrollment_current AS enrollment_remaining
    FROM sections
    ORDER BY enrollment_remaining DESC
    LIMIT 10
"""

pl.read_database(query, connection=engine)

## Running the Pipeline Again

In [ ]:
tables["terms"].write_database("terms", connection=engine, if_table_exists="append")

## Upserting Changed Rows

### Upserting a DataFrame

In [ ]:
changes = pl.DataFrame({
    "term_code":          [202601, 202601],
    "crn":                [10158, 10771],
    "subject_code":       ["DAT", "DAT"],
    "course_number":      ["153", "153"],
    "section":            ["A", "B"],
    "course_title":       ["Database Programming", "Database Programming"],
    "credits":            [1, 1],
    "enrollment_current": [22, 3],
    "enrollment_max":     [24, 24],
})

In [ ]:
with engine.begin() as conn:
    conn.execute(text("CREATE TEMP TABLE sections_staging (LIKE sections) ON COMMIT DROP"))
    changes.write_database("sections_staging", connection=conn, if_table_exists="append")

    result = conn.execute(text("""
        INSERT INTO sections
        SELECT * FROM sections_staging
        ON CONFLICT (term_code, crn) DO UPDATE
        SET enrollment_current = EXCLUDED.enrollment_current,
            enrollment_max     = EXCLUDED.enrollment_max
    """))
    print(f"{result.rowcount} rows inserted or updated")

In [ ]:
pl.read_database("""
    SELECT crn, subject_code, course_number, section, enrollment_current, enrollment_max
    FROM sections
    WHERE subject_code = 'DAT' AND course_number = '153'
    ORDER BY section
""", connection=engine)

## Putting It All Together

In [ ]:
import os

import polars as pl
from dotenv import find_dotenv, load_dotenv
from sqlalchemy import URL, create_engine, text

# ── Config ────────────────────────────────────────────────────────────────────
COURSES_CSV = "data/courses_fall2026.csv"

load_dotenv(find_dotenv(usecwd=True))
engine = create_engine(URL.create(
    "postgresql+psycopg2",
    username=os.environ["PGUSER"],
    password=os.environ["PGPASSWORD"],
    host=os.environ["PGHOST"],
    port=int(os.environ["PGPORT"]),
    database="course_etl",
))

# ── Extract ───────────────────────────────────────────────────────────────────
print("Extracting...")
raw = pl.read_csv(COURSES_CSV, infer_schema=False)

# ── Transform: clean ──────────────────────────────────────────────────────────
print("Cleaning...")
flat = (
    raw.with_columns(pl.col(pl.String).str.strip_chars())
    .with_columns(
        pl.col(
            "term_code", "crn", "credits",
            "enrollment_current", "enrollment_max", "meeting_number",
        ).cast(pl.Int64),
        pl.col("start_time").str.to_time("%H%M"),
        pl.col("end_time").str.to_time("%H%M"),
        pl.col("instructors").str.split("; "),
    )
    .drop("enrollment_remaining", "departments")
    .explode("instructors", empty_as_null=True)
    .with_columns(
        pl.col("instructors")
          .str.split_exact(", ", 1)
          .struct.rename_fields(["instructor_last_name", "instructor_first_name"])
          .alias("instructor_name")
    )
    .unnest("instructor_name")
    .drop("instructors")
)

# ── Transform: normalize ──────────────────────────────────────────────────────
print("Normalizing...")
instructors = (
    flat.select("instructor_first_name", "instructor_last_name")
    .drop_nulls().unique().sort("instructor_last_name", "instructor_first_name")
    .with_row_index("instructor_id", offset=1)
    .with_columns(pl.col("instructor_id").cast(pl.Int64))
)

tables = {
    "terms": flat.select("term_code", "term_description").unique(),
    "subjects": flat.select("subject_code", "subject_description").unique(),
    "buildings": (
        flat.select("building_code", "building_description")
            .drop_nulls("building_code").unique()
    ),
    "courses": flat.select("subject_code", "course_number").unique(),
    "sections": flat.select(
        "term_code", "crn", "subject_code", "course_number", "section",
        "course_title", "credits", "enrollment_current", "enrollment_max",
    ).unique(),
    "meetings": flat.select(
        "term_code", "crn", "meeting_number", "meeting_type",
        "weekdays", "start_time", "end_time", "building_code", "room",
    ).unique(),
    "instructors": instructors,
    "section_instructors": (
        flat.select("term_code", "crn", "instructor_first_name", "instructor_last_name")
            .drop_nulls().unique()
            .join(instructors, on=["instructor_first_name", "instructor_last_name"], how="inner")
            .select("term_code", "crn", "instructor_id")
    ),
}

# ── Load ──────────────────────────────────────────────────────────────────────
print("Loading...")
with engine.begin() as conn:
    conn.execute(text("TRUNCATE " + ", ".join(tables)))
    for name, df in tables.items():
        df.write_database(name, connection=conn, if_table_exists="append")
        print(f"  {name:<22}{len(df):>6,} rows")

print("Done.")

## Activity

**1.** Run the load from Step 4 twice, and confirm the row counts don't change. Then remove the `TRUNCATE` line and run it again. Which table fails, and why that one?

**2.** Simulate a failure partway through a load: inside the loop in Step 4, add `if name == "meetings": raise RuntimeError("connection lost")` after the `write_database()` call. Run it, then check the row counts. What state is the database in? Now predict what would have happened if each table had been loaded in its own transaction.

**3.** Write a SQL query that lists every instructor who teaches in more than one subject, with their subject codes in a single comma-separated column. (Look up PostgreSQL's `STRING_AGG()`.)

**4.** Write a SQL query that finds the ten most over-enrolled sections, showing each section's title, its instructors' names, and how far over its maximum it is. Leave out sections with no fixed cap, where `enrollment_max` is 0.

**5.** Write a SQL query that totals the scheduled minutes per week in each building. A meeting's weekly minutes are its length times the number of days in `weekdays`. Which building is busiest?